In [1]:
import pandas as pd
from io import StringIO

with open("data.csv", "r") as f:
    lines = [line.strip().strip('"') for line in f]

csv_text = "\n".join(lines)

df = pd.read_csv(StringIO(csv_text))

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (10000, 7)
Columns: ['timestamp', 'open', 'high', 'low', 'close', 'volume_btc', 'volume_usd']


In [2]:
import yaml

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

required_fields = ["seed", "window", "version"]

for field in required_fields:
    if field not in config:
        raise ValueError(f"Missing required config field: {field}")

print("Config Loaded Successfully")
print(config)

Config Loaded Successfully
{'seed': 42, 'window': 5, 'version': 'v1'}


In [3]:
required_column = "close"

if df.empty:
    raise ValueError("Dataset is empty")

if required_column not in df.columns:
    raise ValueError(f"Missing required column: {required_column}")

print("Dataset Validation Passed")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset Validation Passed
Rows: 10000
Columns: 7


In [4]:
window = config["window"]

df["rolling_mean"] = df["close"].rolling(window=window).mean()

df[["close", "rolling_mean"]].head(10)

,close,rolling_mean
0,45024.68,NaN
1,45017.83,NaN
2,45050.03,NaN
3,45125.82,NaN
4,45114.17,45066.506
5,45102.53,45082.076
6,45181.21,45114.752
7,45219.50,45148.646
8,45196.09,45162.700
9,45223.17,45184.500


In [5]:
df["signal"] = (df["close"] > df["rolling_mean"]).astype(int)

df[["close", "rolling_mean", "signal"]].head(10)

,close,rolling_mean,signal
0,45024.68,NaN,0
1,45017.83,NaN,0
2,45050.03,NaN,0
3,45125.82,NaN,0
4,45114.17,45066.506,1
5,45102.53,45082.076,1
6,45181.21,45114.752,1
7,45219.50,45148.646,1
8,45196.09,45162.700,1
9,45223.17,45184.500,1


In [6]:
import time

start_time = time.time()

rows_processed = len(df)

signal_rate = df["signal"].mean()

latency_ms = int((time.time() - start_time) * 1000)

print("rows_processed:", rows_processed)
print("signal_rate:", signal_rate)
print("latency_ms:", latency_ms)

rows_processed: 10000
signal_rate: 0.4989
latency_ms: 5


In [7]:
metrics = {
    "version": config["version"],
    "rows_processed": rows_processed,
    "metric": "signal_rate",
    "value": round(float(signal_rate), 4),
    "latency_ms": latency_ms,
    "seed": config["seed"],
    "status": "success"
}

metrics

{'version': 'v1',
 'rows_processed': 10000,
 'metric': 'signal_rate',
 'value': 0.4989,
 'latency_ms': 5,
 'seed': 42,
 'status': 'success'}

In [8]:
import json

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("metrics.json created successfully")

metrics.json created successfully


In [9]:
with open("metrics.json", "r") as f:
    print(f.read())

{
  "version": "v1",
  "rows_processed": 10000,
  "metric": "signal_rate",
  "value": 0.4989,
  "latency_ms": 5,
  "seed": 42,
  "status": "success"
}


In [10]:
import logging

logging.basicConfig(
    filename="run.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Job Started")
logging.info(f"Config Loaded: {config}")
logging.info(f"Rows Loaded: {len(df)}")
logging.info("Rolling Mean Computed")
logging.info("Signals Generated")
logging.info(f"Metrics: {metrics}")
logging.info("Job Completed Successfully")

print("run.log created successfully")

run.log created successfully


In [11]:
with open("run.log", "r") as f:
    print(f.read())

2026-06-01 20:51:32,068 - INFO - Job Started
2026-06-01 20:51:32,069 - INFO - Config Loaded: {'seed': 42, 'window': 5, 'version': 'v1'}
2026-06-01 20:51:32,069 - INFO - Rows Loaded: 10000
2026-06-01 20:51:32,069 - INFO - Rolling Mean Computed
2026-06-01 20:51:32,070 - INFO - Signals Generated
2026-06-01 20:51:32,070 - INFO - Metrics: {'version': 'v1', 'rows_processed': 10000, 'metric': 'signal_rate', 'value': 0.4989, 'latency_ms': 5, 'seed': 42, 'status': 'success'}
2026-06-01 20:51:32,070 - INFO - Job Completed Successfully

